# DINO — Reduced Reproduction on STL-10

**Paper:** Caron et al., *Emerging Properties in Self-Supervised Vision Transformers*, ICCV 2021  
**arXiv:** https://arxiv.org/abs/2104.14294  
**Official code:** https://github.com/facebookresearch/dino

## Scope of this reproduction
| | Paper | This notebook |
|---|---|---|
| Dataset | ImageNet (1.2M images) | STL-10 unlabeled (100K images) |
| Architecture | ViT-S/16 or ViT-B/8 (21–85M params) | ViT-Tiny/8 (~5M params) |
| Training | 300 epochs, bs=1024, 8×V100 | 100 epochs, bs=256, 1×T4 |
| Multi-crop | 2×224² + 6×96² | 2×96² global + 4×48² local |
| Evaluation | Linear probe + k-NN on ImageNet | k-NN on STL-10 test set |

## Notebook structure
1. Setup & dependencies
2. Dataset & augmentations (multi-crop)
3. Model — ViT-Tiny backbone + projection head
4. DINO loss (cross-entropy with centering + sharpening)
5. Training loop (student + EMA teacher)
6. Checkpointing
7. k-NN evaluation
8. Attention map visualisation
9. Ablation experiments (A1: no momentum / A2: no multi-crop / A3: temperature sweep)

---
## 1. Setup & Dependencies

In [ ]:
# Install / verify dependencies
!pip install torch torchvision timm einops --quiet

import os
import math
import copy
import random
import numpy as np
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
import timm

import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device: CUDA (Colab) -> Apple MPS (Mac GPU) -> CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 1b. Run configuration (FAST_MODE)

In [ ]:
# ============================================================
#  RUN CONFIGURATION  —  read this before launching
# ============================================================
# FAST_MODE = True  -> deadline-feasible reduced run.
#   Full pipeline (main pretrain + supervised baseline + A1 + A2 + A3 sweep)
#   targets ~2-3h on a Colab T4. This is an HONEST reduction of the paper
#   setting (documented in decisions.md / the report's Cadre de reproduction).
# FAST_MODE = False -> paper-faithful settings from decisions.md (~8-12h,
#   needs multiple Colab sessions + checkpoint resume).
FAST_MODE = True

if FAST_MODE:
    N_UNLABELED     = 20000      # subsample of the 100k unlabeled split
    N_EPOCHS        = 40         # main DINO pretraining epochs
    ABLATION_EPOCHS = 20         # epochs per ablation run
    OUT_DIM         = 2048       # DINO head output dim
    A3_TAUS         = [0.04, 0.07, 0.1]   # teacher-temp sweep (reduced)
    SUP_EPOCHS      = 30         # supervised baseline epochs
else:
    N_UNLABELED     = 100000
    N_EPOCHS        = 100
    ABLATION_EPOCHS = 50
    OUT_DIM         = 4096
    A3_TAUS         = [0.02, 0.04, 0.07, 0.1]
    SUP_EPOCHS      = 50

N_GLOBAL      = 2
N_LOCAL       = 4
BATCH_SIZE    = 256
WARMUP_EPOCHS = 5
BASE_LR       = 0.0005 * BATCH_SIZE / 256   # linear scaling rule (paper)
WEIGHT_DECAY  = 0.04
MOMENTUM_BASE = 0.996                        # EMA teacher momentum start
MOMENTUM_END  = 1.0                          # EMA teacher momentum end
TAU_S         = 0.1                          # student temperature
TAU_T         = 0.04                         # teacher temperature
KNN_K         = 20
NUM_WORKERS   = 2 if DEVICE.type == 'cuda' else 0  # 0 avoids Jupyter/macOS worker hangs

CHECKPOINT_DIR = './checkpoints'
RESULTS_DIR    = './results'
FIGURES_DIR    = './figures'
for _d in (CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR):
    os.makedirs(_d, exist_ok=True)

print(f'FAST_MODE={FAST_MODE} | unlabeled={N_UNLABELED} | main_epochs={N_EPOCHS} | '
      f'ablation_epochs={ABLATION_EPOCHS} | out_dim={OUT_DIM} | A3_taus={A3_TAUS}')


---
## 2. Dataset & Augmentations

STL-10 specifics:
- **Unlabeled split**: 100,000 images at 96×96 — used for DINO pretraining
- **Train split**: 5,000 labeled images — used to build k-NN reference bank
- **Test split**: 8,000 labeled images — used for k-NN evaluation

Multi-crop strategy (simplified from paper):
- 2 global crops at 96×96 (≥50% of image) — passed to both student and teacher
- 4 local crops at 48×48 (<50% of image) — passed to student only

> **Ablation A2** will disable local crops and use 2 global crops only.

In [ ]:
# ── Augmentation pipelines ──
MEAN = (0.4467, 0.4398, 0.4066)  # STL-10 channel means
STD  = (0.2603, 0.2566, 0.2713)  # STL-10 channel stds

def make_global_transform(img_size=96):
    """Large crop — passed to both student and teacher."""
    return T.Compose([
        T.RandomResizedCrop(img_size, scale=(0.4, 1.0), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
        T.RandomApply([T.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
        T.RandomGrayscale(p=0.2),
        T.RandomApply([T.GaussianBlur(kernel_size=9)], p=0.5),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])

def make_local_transform(img_size=48):
    """Small crop — passed to student only (local-to-global objective)."""
    return T.Compose([
        T.RandomResizedCrop(img_size, scale=(0.05, 0.4), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
        T.RandomApply([T.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
        T.RandomGrayscale(p=0.2),
        T.RandomApply([T.GaussianBlur(kernel_size=5)], p=0.5),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])

class MultiCropTransform:
    """Generates n_global global views + n_local local views from one image."""
    def __init__(self, n_global=2, n_local=4, use_local=True):
        self.global_tf = make_global_transform(96)
        self.local_tf  = make_local_transform(48)
        self.n_global  = n_global
        self.n_local   = n_local if use_local else 0
    def __call__(self, img):
        crops  = [self.global_tf(img) for _ in range(self.n_global)]
        crops += [self.local_tf(img)  for _ in range(self.n_local)]
        return crops

# ── Datasets ──
DATA_DIR = './data'

pretrain_full = torchvision.datasets.STL10(
    root=DATA_DIR, split='unlabeled', download=True,
    transform=MultiCropTransform(n_global=N_GLOBAL, n_local=N_LOCAL, use_local=True),
)
# Deterministic subsample for FAST_MODE (and reused by ablations)
_g = torch.Generator().manual_seed(SEED)
PRETRAIN_IDX_FULL = torch.randperm(len(pretrain_full), generator=_g).tolist()  # full shuffle
PRETRAIN_IDX = PRETRAIN_IDX_FULL[:N_UNLABELED]
pretrain_dataset = torch.utils.data.Subset(pretrain_full, PRETRAIN_IDX) \
    if N_UNLABELED < len(pretrain_full) else pretrain_full

pretrain_loader = DataLoader(
    pretrain_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)

eval_transform = T.Compose([
    T.Resize(96, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(96),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
train_dataset = torchvision.datasets.STL10(root=DATA_DIR, split='train', download=True, transform=eval_transform)
test_dataset  = torchvision.datasets.STL10(root=DATA_DIR, split='test',  download=True, transform=eval_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Pretrain: {len(pretrain_dataset):,} images (from {len(pretrain_full):,} unlabeled)')
print(f'k-NN ref: {len(train_dataset):,} | k-NN eval: {len(test_dataset):,}')


---
## 3. Model — ViT-Tiny Backbone + Projection Head

Architecture choices:
- **Backbone**: `vit_tiny_patch16_224` from `timm`, adapted to patch_size=8 and img_size=96
- **Projection head**: 3-layer MLP (hidden dim 2048, L2-normalised output of dim K=4096)
- Both student and teacher share the same architecture; teacher weights are never directly optimised

In [ ]:
# ── Projection head ──
class DINOHead(nn.Module):
    """3-layer MLP projection head as in the paper (Appendix C)."""
    def __init__(self, in_dim, out_dim=4096, hidden_dim=2048, bottleneck_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, bottleneck_dim),
        )
        self.last_layer = nn.utils.weight_norm(nn.Linear(bottleneck_dim, out_dim, bias=False))
        self.last_layer.weight_g.data.fill_(1)
        self.last_layer.weight_g.requires_grad = False
    def forward(self, x):
        x = self.mlp(x)
        x = F.normalize(x, dim=-1, p=2)
        x = self.last_layer(x)
        return x

# ── Full DINO network (backbone + head) ──
class DINONet(nn.Module):
    def __init__(self, out_dim=4096):
        super().__init__()
        self.backbone = timm.create_model(
            'vit_tiny_patch16_224', pretrained=False,
            img_size=96, patch_size=8, num_classes=0,
        )
        embed_dim = self.backbone.embed_dim  # 192 for ViT-Tiny
        self.head = DINOHead(embed_dim, out_dim=out_dim)
    def forward(self, x):
        return self.head(self.backbone(x))

def build_pair(out_dim):
    """Create a fresh (student, teacher) pair; teacher = frozen copy of student."""
    s = DINONet(out_dim=out_dim).to(DEVICE)
    t = DINONet(out_dim=out_dim).to(DEVICE)
    t.load_state_dict(s.state_dict())
    for p in t.parameters():
        p.requires_grad = False
    return s, t

student, teacher = build_pair(OUT_DIM)
print(f'Student parameters: {sum(p.numel() for p in student.parameters())/1e6:.1f}M')


---
## 4. DINO Loss

Cross-entropy loss between teacher (centered + sharpened) and student distributions.

- **Centering**: running mean subtracted from teacher output to prevent dimensional collapse
- **Sharpening**: low temperature `tau_t` on teacher softmax to produce peaked distributions
- **Stop-gradient**: no gradient flows through teacher

> **Ablation A3** sweeps `tau_t` ∈ {0.02, 0.04, 0.07, 0.1}

In [ ]:
class DINOLoss(nn.Module):
    """DINO self-distillation loss (Eq. 3): cross-entropy between centered+sharpened
    teacher and student distributions, over all (teacher_view, student_view) pairs
    with differing views. Center updated by EMA (Eq. 4)."""
    def __init__(self, out_dim, n_global=2, n_local=4, tau_s=0.1, tau_t=0.04, center_momentum=0.9):
        super().__init__()
        self.n_global=n_global; self.n_local=n_local
        self.tau_s=tau_s; self.tau_t=tau_t; self.center_momentum=center_momentum
        self.register_buffer('center', torch.zeros(1, out_dim))
    def forward(self, student_output, teacher_output):
        teacher_out = [F.softmax((t - self.center) / self.tau_t, dim=-1).detach() for t in teacher_output]
        student_out = [F.log_softmax(s / self.tau_s, dim=-1) for s in student_output]
        total_loss, n_pairs = 0.0, 0
        for i, t in enumerate(teacher_out):
            for j, s in enumerate(student_out):
                if i == j:
                    continue
                total_loss += -(t * s).sum(dim=-1).mean()
                n_pairs += 1
        total_loss /= n_pairs
        self.update_center(teacher_output)
        return total_loss
    @torch.no_grad()
    def update_center(self, teacher_output):
        batch_center = torch.cat(teacher_output).mean(dim=0, keepdim=True)
        self.center = self.center * self.center_momentum + batch_center * (1 - self.center_momentum)

dino_loss = DINOLoss(OUT_DIM, n_global=N_GLOBAL, n_local=N_LOCAL, tau_s=TAU_S, tau_t=TAU_T).to(DEVICE)
print('DINO loss initialised.')


---
## 5. Training Loop

Key components:
- **Optimiser**: AdamW with cosine LR schedule and linear warmup (5 epochs)
- **EMA teacher update**: `theta_t ← lambda * theta_t + (1 - lambda) * theta_s`
  - `lambda` follows a cosine schedule from 0.996 → 1.0
- Only global crops are passed to the teacher; all crops go to the student

> **Ablation A1** replaces EMA update with direct copy: `teacher = copy(student)`

In [ ]:
# ── Schedules (horizon-aware so ablations decay correctly) ──
def get_lr(epoch, n_epochs=None, base_lr=None):
    n_epochs = N_EPOCHS if n_epochs is None else n_epochs
    base_lr  = BASE_LR  if base_lr  is None else base_lr
    if epoch < WARMUP_EPOCHS:
        return base_lr * (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, (n_epochs - WARMUP_EPOCHS))
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))

def get_momentum(epoch, n_epochs=None):
    n_epochs = N_EPOCHS if n_epochs is None else n_epochs
    progress = epoch / n_epochs
    return MOMENTUM_END - (MOMENTUM_END - MOMENTUM_BASE) * (math.cos(math.pi * progress) + 1) / 2

@torch.no_grad()
def update_teacher_ema(student, teacher, momentum):
    for ps, pt in zip(student.parameters(), teacher.parameters()):
        pt.data = momentum * pt.data + (1.0 - momentum) * ps.data

def train_one_epoch(student, teacher, loader, loss_fn, optimizer, epoch,
                    n_epochs=None, use_momentum=True):
    student.train(); teacher.eval()
    total_loss = 0.0
    lr = get_lr(epoch, n_epochs)
    for g in optimizer.param_groups:
        g['lr'] = lr
    momentum = get_momentum(epoch, n_epochs)
    for crops, _ in loader:
        crops = [c.to(DEVICE, non_blocking=True) for c in crops]
        global_crops = crops[:N_GLOBAL]            # teacher sees globals only
        with torch.no_grad():
            teacher_output = [teacher(c) for c in global_crops]
        student_output = [student(c) for c in crops]   # student sees all crops
        loss = loss_fn(student_output, teacher_output)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), max_norm=3.0)
        optimizer.step()
        if use_momentum:
            update_teacher_ema(student, teacher, momentum)
        else:
            teacher.load_state_dict(student.state_dict())   # Ablation A1
        total_loss += loss.item()
    return total_loss / len(loader)

optimizer = torch.optim.AdamW(student.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
print('Training utilities defined.')


In [ ]:
# ── Run main DINO pretraining ──
import time
loss_history = []
_t0 = time.time()
for epoch in range(N_EPOCHS):
    avg = train_one_epoch(student, teacher, pretrain_loader, dino_loss, optimizer,
                          epoch, n_epochs=N_EPOCHS, use_momentum=True)
    loss_history.append(avg)
    print(f'Epoch [{epoch+1:3d}/{N_EPOCHS}] loss {avg:.4f}  lr {get_lr(epoch):.6f}  '
          f'ema_m {get_momentum(epoch):.4f}  ({(time.time()-_t0)/60:.1f} min)')
    if (epoch+1) % 10 == 0 or (epoch+1) == N_EPOCHS:
        path = os.path.join(CHECKPOINT_DIR, f'dino_epoch{epoch+1:03d}.pt')
        torch.save({'epoch':epoch+1,'student':student.state_dict(),
                    'teacher':teacher.state_dict(),'optimizer':optimizer.state_dict(),
                    'loss_history':loss_history}, path)
        print(f'  -> checkpoint: {path}')

# Loss curve figure for the report
plt.figure(figsize=(6,4))
plt.plot(range(1,len(loss_history)+1), loss_history)
plt.xlabel('epoch'); plt.ylabel('DINO loss'); plt.title('Main pretraining loss')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR,'loss_curve.png'), dpi=150, bbox_inches='tight'); plt.show()


---
## 5b. (Recommandé) Entraînement adaptatif piloté par la métrique aval

Plutôt que de fixer arbitrairement la taille du jeu non étiqueté (12k ? 20k ?), on **fait
croître** le pool par tranches de `CHUNK` images. Après chaque tranche on entraîne quelques
époques puis on évalue un **k-NN sonde** sur un petit échantillon tenu à l'écart. On **arrête
automatiquement** quand la précision sonde plafonne (gain < `min_delta` pendant `patience`
phases) ou atteint une cible. Le critère d'arrêt devient ainsi *empirique* et non arbitraire, et
la courbe « précision vs. nombre d'images » constitue notre extension personnelle.

> Exécuter **soit** la cellule §5 (budget fixe), **soit** celle-ci. Après l'entraînement, le
> meilleur modèle est restauré dans `student`/`teacher` ; lancer ensuite §7 pour le k-NN *complet*
> (5\,000 réf. / 8\,000 test), qui est le chiffre rapporté (la sonde n'est qu'un signal rapide).

In [ ]:
# ── Adaptive training: grow data + early-stop on downstream k-NN ──
from torch.utils.data import Subset

# ── Eval helpers (defined here so the in-training probe can use them; reused in §7) ──
@torch.no_grad()
def extract_features(model, loader):
    """Extract L2-normalised CLS-token features and labels for a dataset.
    `model` must expose `.backbone(imgs) -> (B, D)`."""
    model.eval()
    feats, labels = [], []
    for imgs, lbs in loader:
        f = model.backbone(imgs.to(DEVICE))
        f = F.normalize(f, dim=-1)
        feats.append(f.cpu()); labels.append(lbs)
    return torch.cat(feats), torch.cat(labels)

@torch.no_grad()
def knn_accuracy(train_features, train_labels, test_features, test_labels,
                 k=20, temp=0.07, num_classes=10):
    """Weighted k-NN (temperature-scaled cosine similarity)."""
    correct = 0
    tf = train_features.to(DEVICE); tl = train_labels.to(DEVICE)
    for i in range(0, len(test_features), 256):
        bf = test_features[i:i+256].to(DEVICE)
        bl = test_labels[i:i+256].to(DEVICE)
        sim = bf @ tf.T
        sim_topk, idx_topk = sim.topk(k, dim=1)
        weights = F.softmax(sim_topk / temp, dim=1)
        neigh = tl[idx_topk]
        scores = torch.zeros(len(bf), num_classes, device=DEVICE)
        scores.scatter_add_(1, neigh, weights)
        correct += (scores.argmax(1) == bl).sum().item()
    return 100.0 * correct / len(test_features)


# Small fixed probe sets for the FAST in-training probe (NOT the final number).
# Carved from DISJOINT slices of the labeled TRAIN split so the TEST set is NEVER
# used for early-stopping / model selection (avoids optimistic test-set bias).
PROBE_REF_N, PROBE_Q_N = 1000, 1000
_probe_ref = DataLoader(Subset(train_dataset, list(range(0, PROBE_REF_N))),
                        batch_size=256, shuffle=False, num_workers=NUM_WORKERS)
_probe_q   = DataLoader(Subset(train_dataset, list(range(PROBE_REF_N, PROBE_REF_N + PROBE_Q_N))),
                        batch_size=256, shuffle=False, num_workers=NUM_WORKERS)

@torch.no_grad()
def probe_knn(model):
    rf, rl = extract_features(model, _probe_ref)
    qf, ql = extract_features(model, _probe_q)
    return knn_accuracy(rf, rl, qf, ql, k=KNN_K)

def train_dino_adaptive(chunk=4000, epochs_per_phase=10, max_images=24000,
                        target_acc=None, patience=2, min_delta=1.0):
    """Grow the unlabeled pool by `chunk` images each phase, train `epochs_per_phase`
    epochs on the accumulated pool, then probe k-NN on a small held-out sample.
    Stop when probe accuracy gains < `min_delta` pp for `patience` phases, reaches
    `target_acc`, or the pool hits `max_images`/dataset size. Restores the best model."""
    global student, teacher, optimizer, dino_loss
    student, teacher = build_pair(OUT_DIM)
    dino_loss = DINOLoss(OUT_DIM, n_global=N_GLOBAL, n_local=N_LOCAL, tau_s=TAU_S, tau_t=TAU_T).to(DEVICE)
    optimizer = torch.optim.AdamW(student.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)

    cap = min(max_images, len(pretrain_full))
    n_phases_max = math.ceil(cap / chunk)
    horizon = n_phases_max * epochs_per_phase          # cosine LR/momentum horizon
    history, best_acc, best_state, no_improve, global_epoch = [], -1.0, None, 0, 0

    for phase in range(1, n_phases_max + 1):
        n_img = min(phase * chunk, cap)
        loader = DataLoader(Subset(pretrain_full, PRETRAIN_IDX_FULL[:n_img]),
                            batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                            pin_memory=(DEVICE.type == 'cuda'), drop_last=True)
        print(f'\n── Phase {phase}/{n_phases_max} | pool = {n_img} images ──')
        for _ in range(epochs_per_phase):
            avg = train_one_epoch(student, teacher, loader, dino_loss, optimizer,
                                  global_epoch, n_epochs=horizon, use_momentum=True)
            global_epoch += 1
        acc = probe_knn(teacher)
        history.append((n_img, global_epoch, acc))
        print(f'  loss {avg:.4f} | probe k-NN ({PROBE_Q_N} q) after {global_epoch} ep: {acc:.2f}%')

        if acc > best_acc:
            best_acc = acc
            best_state = {'student': {k: v.detach().cpu().clone() for k, v in student.state_dict().items()},
                          'teacher': {k: v.detach().cpu().clone() for k, v in teacher.state_dict().items()},
                          'images': n_img, 'epoch': global_epoch}
            torch.save(best_state, os.path.join(CHECKPOINT_DIR, 'dino_adaptive_best.pt'))

        gain = acc - history[-2][2] if len(history) >= 2 else float('inf')
        no_improve = no_improve + 1 if gain < min_delta else 0
        if target_acc is not None and acc >= target_acc:
            print(f'  reached target {target_acc}% -> stop'); break
        if no_improve >= patience:
            print(f'  plateau ({patience} phases gain < {min_delta}pp) -> stop'); break

    if best_state is not None:
        student.load_state_dict({k: v.to(DEVICE) for k, v in best_state['student'].items()})
        teacher.load_state_dict({k: v.to(DEVICE) for k, v in best_state['teacher'].items()})

    with open(os.path.join(RESULTS_DIR, 'adaptive_history.json'), 'w') as f:
        json.dump([{'images': a, 'epoch': b, 'probe_knn': c} for a, b, c in history], f, indent=2)
    xs = [h[0] for h in history]; ys = [h[2] for h in history]
    plt.figure(figsize=(6, 4)); plt.plot(xs, ys, 'o-')
    plt.xlabel('unlabeled images used'); plt.ylabel('probe k-NN top-1 (%)')
    plt.title('Adaptive training — downstream accuracy vs. data')
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'adaptive_curve.png'), dpi=150, bbox_inches='tight'); plt.show()
    print(f'\nBest probe k-NN {best_acc:.2f}% at {best_state["images"]} images / {best_state["epoch"]} ep '
          f'-> best model restored into student/teacher.')
    return history

# Run it (tune target_acc / patience as you like):
adaptive_history = train_dino_adaptive(chunk=4000, epochs_per_phase=10, max_images=24000,
                                       target_acc=None, patience=2, min_delta=1.0)


---
## 6. Load Checkpoint

In [ ]:
def load_checkpoint(path):
    ckpt = torch.load(path, map_location=DEVICE)
    student.load_state_dict(ckpt['student'])
    teacher.load_state_dict(ckpt['teacher'])
    print(f'Loaded checkpoint from epoch {ckpt["epoch"]} (loss: {ckpt["loss_history"][-1]:.4f})')
    return ckpt['loss_history']

# Example: load_checkpoint('./checkpoints/dino_epoch100.pt')

---
## 7. k-NN Evaluation

Protocol:
1. Extract frozen CLS-token features from teacher backbone over STL-10 train set (reference bank)
2. For each test image, find its k=20 nearest neighbors in the reference bank
3. Majority-vote the labels of those neighbors → predicted class
4. Report top-1 accuracy on the 8,000 test images

In [ ]:
# Final k-NN on the FULL held-out TEST set (the reported number).
# extract_features / knn_accuracy are defined in §5b (adaptive cell).
print('Extracting DINO (teacher) features...')
train_feats, train_lbs = extract_features(teacher, train_loader)
test_feats,  test_lbs  = extract_features(teacher, test_loader)
dino_knn = knn_accuracy(train_feats, train_lbs, test_feats, test_lbs, k=KNN_K)
print(f'DINO teacher   k-NN top-1: {dino_knn:.2f}%')

# Lower-bound: random-init backbone
rand_s, _ = build_pair(OUT_DIM)
rf, rl = extract_features(rand_s, train_loader)
ef, el = extract_features(rand_s, test_loader)
random_knn = knn_accuracy(rf, rl, ef, el, k=KNN_K)
print(f'Random-init    k-NN top-1: {random_knn:.2f}%')

main_results = {'dino_knn': dino_knn, 'random_knn': random_knn}


---
## 7b. Supervised baseline (apples-to-apples)

Per `decisions.md` §5: a ViT-Tiny of the **same architecture**, trained from scratch with
supervised cross-entropy on STL-10's 5,000 labeled train images, then evaluated with the
**same k-NN protocol** on its frozen backbone features. We also report its direct test
classification accuracy. This isolates the contribution of *self-supervision*: same model,
same eval, supervised vs. SSL.

In [ ]:
class _Holder(nn.Module):
    """Wrap a classifier so extract_features() can call `.backbone(x)` -> features."""
    def __init__(self, clf):
        super().__init__(); self.clf = clf
    def backbone(self, x):
        f = self.clf.forward_features(x)
        return self.clf.forward_head(f, pre_logits=True)

def train_supervised_baseline(n_epochs):
    model = timm.create_model('vit_tiny_patch16_224', pretrained=False,
                              img_size=96, patch_size=8, num_classes=10).to(DEVICE)
    sup_tf = T.Compose([
        T.RandomResizedCrop(96, scale=(0.5,1.0), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(MEAN, STD)])
    sup_ds = torchvision.datasets.STL10(root=DATA_DIR, split='train', download=False, transform=sup_tf)
    sup_loader = DataLoader(sup_ds, batch_size=128, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)
    crit = nn.CrossEntropyLoss()
    for ep in range(n_epochs):
        model.train(); tot = 0.0
        for x, y in sup_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = crit(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        if (ep+1) % 5 == 0:
            print(f'  sup epoch {ep+1}/{n_epochs} loss {tot/len(sup_loader):.3f}')
    return model

@torch.no_grad()
def classification_accuracy(model, loader):
    model.eval(); correct = tot = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model(x).argmax(1) == y).sum().item(); tot += y.numel()
    return 100.0 * correct / tot

print('Training supervised baseline...')
sup_model = train_supervised_baseline(SUP_EPOCHS)
sup_test_acc = classification_accuracy(sup_model, test_loader)
print(f'Supervised baseline  test-cls top-1: {sup_test_acc:.2f}%')

sup_holder = _Holder(sup_model).to(DEVICE)
sf, sl = extract_features(sup_holder, train_loader)
sef, sel = extract_features(sup_holder, test_loader)
sup_knn = knn_accuracy(sf, sl, sef, sel, k=KNN_K)
print(f'Supervised baseline  k-NN top-1:     {sup_knn:.2f}%')

main_results.update({'supervised_knn': sup_knn, 'supervised_test_cls': sup_test_acc})
with open(os.path.join(RESULTS_DIR,'main_results.json'),'w') as f:
    json.dump(main_results, f, indent=2)
print('Saved results/main_results.json:', main_results)


---
## 8. Attention Map Visualisation

Visualise the self-attention of the [CLS] token in the last ViT block.
Different heads should attend to different semantic regions of the image — the paper's key qualitative result (Figure 1 & 3).

In [ ]:
@torch.no_grad()
def visualise_attention(model, img_tensor, patch_size=8, save_path=None, title=''):
    """Per-head self-attention of the [CLS] token in the last ViT block."""
    model.eval()
    x = img_tensor.unsqueeze(0).to(DEVICE)
    H, W = x.shape[-2:]
    nh, nw = H // patch_size, W // patch_size
    store = {}
    def attn_hook(module, inp, out):
        B, N, C = inp[0].shape
        qkv = module.qkv(inp[0]).reshape(B, N, 3, module.num_heads, C // module.num_heads).permute(2,0,3,1,4)
        q, k, _ = qkv.unbind(0)
        scale = (C // module.num_heads) ** -0.5
        attn = ((q @ k.transpose(-2,-1)) * scale).softmax(dim=-1)
        store['attn'] = attn.detach().cpu()
    h = model.backbone.blocks[-1].attn.register_forward_hook(attn_hook)
    _ = model.backbone(x)
    h.remove()
    attn = store['attn'][0, :, 0, 1:].reshape(-1, nh, nw)          # (heads, nh, nw)
    attn_up = F.interpolate(attn.unsqueeze(0).float(), size=(H,W),
                            mode='bilinear', align_corners=False)[0]
    mean = torch.tensor(MEAN).view(3,1,1); std = torch.tensor(STD).view(3,1,1)
    img = (img_tensor*std+mean).clamp(0,1).permute(1,2,0).numpy()
    nheads = attn_up.shape[0]
    ncol = (nheads + 2 + 1)//2
    fig, axes = plt.subplots(2, ncol, figsize=(2.2*ncol, 5)); axes = axes.flatten()
    axes[0].imshow(img); axes[0].set_title('input'); axes[0].axis('off')
    axes[1].imshow(attn_up.mean(0), cmap='inferno'); axes[1].set_title('mean'); axes[1].axis('off')
    for hd in range(nheads):
        axes[hd+2].imshow(img); axes[hd+2].imshow(attn_up[hd], alpha=0.6, cmap='inferno')
        axes[hd+2].set_title(f'head {hd+1}'); axes[hd+2].axis('off')
    for ax in axes[nheads+2:]: ax.axis('off')
    plt.suptitle(title or 'Self-attention — last block (CLS query)', fontsize=12)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

class_names = ['airplane','bird','car','cat','deer','dog','horse','monkey','ship','truck']
# A few qualitative examples for the report
for n, idx in enumerate([0, 1, 2, 7]):
    img, lab = test_dataset[idx]
    visualise_attention(teacher, img,
                        save_path=os.path.join(FIGURES_DIR, f'attention_{class_names[lab]}.png'),
                        title=f'class: {class_names[lab]}')


---
## 9. Ablation Experiments

| ID | What changes | Expected outcome |
|---|---|---|
| **A1** | No momentum encoder (teacher = direct copy of student) | Collapse → k-NN ≈ random |
| **A2** | No multi-crop (2 global crops only, no local crops) | Drop of ~3–5% in k-NN accuracy |
| **A3** | Teacher temperature sweep τ_t ∈ {0.02, 0.04, 0.07, 0.1} | U-shaped curve, optimum around 0.04 |

In [ ]:
# ── Ablation runner ──
# Re-initialises a fresh model and trains from scratch with ONE change.
ablation_results = {}

def run_ablation(name, use_momentum=True, use_local_crops=True, tau_t=TAU_T,
                 n_epochs=None):
    n_epochs = ABLATION_EPOCHS if n_epochs is None else n_epochs
    print(f'\n── Ablation: {name}  (epochs={n_epochs}) ──')
    s, t = build_pair(OUT_DIM)
    n_local = N_LOCAL if use_local_crops else 0
    abl_full = torchvision.datasets.STL10(
        root=DATA_DIR, split='unlabeled', download=False,
        transform=MultiCropTransform(n_global=N_GLOBAL, n_local=n_local, use_local=use_local_crops))
    abl_ds = torch.utils.data.Subset(abl_full, PRETRAIN_IDX) \
        if N_UNLABELED < len(abl_full) else abl_full
    abl_loader = DataLoader(abl_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    loss_fn = DINOLoss(OUT_DIM, n_global=N_GLOBAL, n_local=n_local, tau_s=TAU_S, tau_t=tau_t).to(DEVICE)
    opt = torch.optim.AdamW(s.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    for epoch in range(n_epochs):
        avg = train_one_epoch(s, t, abl_loader, loss_fn, opt, epoch,
                              n_epochs=n_epochs, use_momentum=use_momentum)
        if (epoch+1) % 5 == 0:
            print(f'  epoch {epoch+1}/{n_epochs}  loss {avg:.4f}')
    tf, tl = extract_features(t, train_loader)
    ef, el = extract_features(t, test_loader)
    acc = knn_accuracy(tf, tl, ef, el, k=KNN_K)
    print(f'  -> k-NN top-1: {acc:.2f}%')
    ablation_results[name] = acc
    return acc

print('Ablation runner defined.')


In [ ]:
# A1 — No momentum encoder (teacher = direct copy of student)
run_ablation('A1_no_momentum', use_momentum=False, use_local_crops=True, tau_t=TAU_T)


In [ ]:
# A2 — No multi-crop (2 global views only, no local crops)
run_ablation('A2_no_multicrop', use_momentum=True, use_local_crops=False, tau_t=TAU_T)


In [ ]:
# A3 — Teacher temperature sweep
for _tau in A3_TAUS:
    run_ablation(f'A3_tau_t={_tau}', use_momentum=True, use_local_crops=True, tau_t=_tau)


In [ ]:
# ── Summary: merge main + baseline + ablations, save for the report ──
print('\n=== Main results ===')
for k, v in main_results.items():
    print(f'{k:<22} {v:.2f}')
print('\n=== Ablation summary ===')
print(f'{"Experiment":<28} k-NN top-1 (%)')
print('-'*44)
for name, acc in ablation_results.items():
    print(f'{name:<28} {acc:.2f}')

all_results = {'config': {'FAST_MODE': FAST_MODE, 'N_UNLABELED': N_UNLABELED,
               'N_EPOCHS': N_EPOCHS, 'ABLATION_EPOCHS': ABLATION_EPOCHS, 'OUT_DIM': OUT_DIM},
               'main': main_results, 'ablations': ablation_results}
with open(os.path.join(RESULTS_DIR,'all_results.json'),'w') as f:
    json.dump(all_results, f, indent=2)
print('\nSaved results/all_results.json')
